# Quantitative Investing - Getting Started

이 노트북은 퀀트 투자 시스템의 기본 사용법을 안내합니다.

## 목차
1. 환경 설정
2. 데이터 수집
3. 전략 생성
4. 백테스팅
5. 성과 분석

## 1. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 한글 폰트 설정 (matplotlib)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("Environment setup complete!")

## 2. 데이터 수집

### 2.1 한국 주식 데이터

In [ ]:
from src.data_collection.kr_stock_collector import KoreanStockCollector

# 수집기 초기화
kr_collector = KoreanStockCollector(use_pykrx=True)

# 주요 종목 (삼성전자, SK하이닉스, NAVER, 카카오, 삼성바이오로직스)
kr_stocks = ['005930', '000660', '035420', '035720', '207940']

# 최근 2년 데이터 수집
end_date = datetime.now()
start_date = end_date - timedelta(days=365*2)

print(f"Collecting data from {start_date.date()} to {end_date.date()}...")

kr_data = kr_collector.collect_multiple_stocks(
    symbols=kr_stocks,
    start_date=start_date,
    end_date=end_date,
    save_to_db=True
)

print(f"\nCollected {len(kr_data)} records for {len(kr_stocks)} stocks")
kr_data.head()

### 2.2 미국 주식 데이터

In [ ]:
from src.data_collection.us_stock_collector import USStockCollector

# 수집기 초기화
us_collector = USStockCollector()

# 주요 기술주
us_stocks = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']

print("Collecting US stock data...")

us_data = us_collector.download_bulk_data(
    symbols=us_stocks,
    start_date=start_date,
    end_date=end_date
)

print(f"\nCollected {len(us_data)} records for {len(us_stocks)} stocks")
us_data.head()

## 3. 전략 생성

### 3.1 모멘텀 전략

In [ ]:
from src.strategies.quant_strategies import create_strategy

# 모멘텀 전략 생성
momentum_strategy = create_strategy('momentum')

print(f"Strategy: {momentum_strategy.name}")
print(f"Config: {momentum_strategy.config}")

### 3.2 멀티팩터 전략

In [ ]:
# 멀티팩터 전략 생성
multifactor_strategy = create_strategy('multifactor')

print(f"Strategy: {multifactor_strategy.name}")
print(f"Factor weights: {multifactor_strategy.weights}")

## 4. 백테스팅

모멘텀 전략으로 백테스트를 실행합니다.

In [ ]:
from src.backtesting.backtester import run_backtest

# 한국 주식으로 백테스트
print("Running backtest...")

results, summary = run_backtest(
    strategy=momentum_strategy,
    data=kr_data,
    start_date=start_date,
    end_date=end_date,
    initial_capital=100000000,  # 1억원
    commission=0.0015,  # 0.15%
    slippage=0.001,  # 0.1%
    rebalance_frequency='monthly'
)

print("\nBacktest completed!")
results.head()

## 5. 성과 분석

### 5.1 성과 요약

In [ ]:
# 성과 요약 출력
print("="*60)
print("BACKTEST SUMMARY")
print("="*60)

for key, value in summary.items():
    if isinstance(value, float):
        if 'return' in key.lower() or 'ratio' in key.lower():
            print(f"{key:.<40} {value:>10.2%}")
        else:
            print(f"{key:.<40} {value:>10.2f}")
    else:
        print(f"{key:.<40} {value:>10}")

print("="*60)

### 5.2 시각화

In [ ]:
from src.backtesting.visualizer import BacktestVisualizer

# 시각화 객체 생성
viz = BacktestVisualizer(results)

# 포트폴리오 가치 추이
viz.plot_portfolio_value()

In [ ]:
# 누적 수익률
viz.plot_cumulative_returns()

In [ ]:
# 낙폭 (Drawdown)
viz.plot_drawdown()

In [ ]:
# 수익률 분포
viz.plot_return_distribution()

In [ ]:
# 월별 수익률 히트맵
viz.plot_monthly_returns_heatmap()

In [ ]:
# 롤링 샤프 비율
viz.plot_rolling_sharpe(window=126)  # 6개월

## 6. 여러 전략 비교

다양한 전략의 성과를 비교해봅시다.

In [ ]:
from src.backtesting.visualizer import compare_strategies_plot
from src.backtesting.performance_metrics import compare_strategies

# 여러 전략 생성
strategies = {
    'Momentum': create_strategy('momentum'),
    'Value': create_strategy('value'),
    'Quality': create_strategy('quality'),
    'Multi-Factor': create_strategy('multifactor')
}

# 각 전략으로 백테스트 실행
results_dict = {}

for name, strategy in strategies.items():
    print(f"\nBacktesting {name} strategy...")
    results, _ = run_backtest(
        strategy=strategy,
        data=kr_data,
        start_date=start_date,
        end_date=end_date,
        initial_capital=100000000,
        rebalance_frequency='monthly'
    )
    results_dict[name] = results

print("\nAll backtests completed!")

In [ ]:
# 전략 비교 테이블
comparison_df = compare_strategies(results_dict)
comparison_df

In [ ]:
# 전략 비교 차트
compare_strategies_plot(results_dict)

## 결론

이 노트북에서는 퀀트 투자 시스템의 기본 사용법을 배웠습니다:

1. 데이터 수집 (한국/미국 주식)
2. 전략 생성 (모멘텀, 밸류, 퀄리티, 멀티팩터)
3. 백테스팅 실행
4. 성과 분석 및 시각화
5. 여러 전략 비교

### 다음 단계

- `02_strategy_development.ipynb`: 나만의 전략 개발하기
- `03_advanced_backtesting.ipynb`: 고급 백테스팅 기법
- `04_risk_management.ipynb`: 리스크 관리 전략

Happy Quant Investing! 📊💰